# 04 · Sınıflandırma — TF-IDF + Naive Bayes / Logistic Regression / Linear SVM

**Final hedefleri ile eşleşme**

| Hocanın maddesi | Bölüm |
|---|---|
| 1 · Sayısal özellik çıkarımı (TF-IDF) | 3 |
| 2 · Birden fazla algoritma (MNB, LR, Linear SVM) | 4 |
| 3 · Eğitim / test ayrımı, görülmemiş belgelerde deneme | 2, 5 |
| 4 · Accuracy, Precision, Recall, F1, confusion matrix, sınıf bazlı inceleme | 5–7 |

**Girdi:** `data/cleaned_dataset.csv` (03'ün çıktısı) · **Çıktı:** `models/best_model.joblib` (05 demo için)

**Veri kullanımı kuralı:** Model ve ayar seçimi sadece **train + validation** ile yapılır. **Test** seti en sonda, bir kez kullanılır.
GPU gerekmez; tüm notebook ~5 dk.

## 1. Kurulum ve veri

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
MODELS = ROOT / "models"; MODELS.mkdir(exist_ok=True)
CLASSES = ["email", "form", "invoice", "resume"]
SEED = 42

df = pd.read_csv(DATA / "cleaned_dataset.csv")
df["text_norm"] = df["text_norm"].fillna("")
print(df.shape)
pd.crosstab(df["label"], df["split"])

## 2. Eğitim / doğrulama / test kümeleri

| Küme | Kullanım | Filtre |
|---|---|---|
| Train | Model öğrenir | OCR'ı tamamen başarısız (< 5 kelime) belgeler **çıkarılır** |
| Validation | Hiperparametre ve model seçimi | Filtre yok |
| Test | Sadece en sonda, bir kez | **Filtre yok** → gerçekçi başarı. Ek olarak okunabilir alt kümede de raporlanır |

Test'ten belge silmek başarıyı yapay olarak yükseltir; bu yüzden test seti olduğu gibi kullanılır.

In [ ]:
train_all = df[df.split == "train"]
train = train_all[~train_all.is_empty].copy()
val = df[df.split == "validation"].copy()
test = df[df.split == "test"].copy()
print(f"Train: {len(train)} (filtrelenen boş belge: {len(train_all) - len(train)})")
print(f"Validation: {len(val)} · Test: {len(test)} (test'te boş belge: {int(test.is_empty.sum())})")

X_tr, y_tr = train["text_norm"], train["label"]
X_va, y_va = val["text_norm"], val["label"]
X_te, y_te = test["text_norm"], test["label"]

## 3. Sayısal özellik çıkarımı: TF-IDF

Her belge, kelimelerinin ağırlıklarından oluşan bir vektöre dönüştürülür:

```latex
\mathrm{tfidf}(t, d) = \mathrm{tf}(t, d) \times \log\frac{1 + N}{1 + \mathrm{df}(t)} + 1
```

- **tf:** kelimenin belgede kaç kez geçtiği · **df:** kaç belgede geçtiği · **N:** belge sayısı
- Her belgede geçen kelimeler (the, and) düşük, belirli belge türlerine özgü kelimeler (invoice, education) yüksek ağırlık alır.

İki çeşit denenir:

| Özellik | Birim | Örnek | Neden |
|---|---|---|---|
| Kelime TF-IDF | Kelime ve kelime ikilileri | `total`, `amount due` | Standart yöntem |
| **Karakter TF-IDF** | 3–5 harflik parçalar | `amo`, `moun`, `ount` | OCR hatalarına dayanıklı: `Anount` ile `Amount` ortak parçalar paylaşır |

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def word_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True,
                           stop_words="english", token_pattern=r"(?u)\b[a-z0][a-z0]+\b")

def char_tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.9,
                           sublinear_tf=True, max_features=100_000)

v = word_tfidf().fit(X_tr)
print("Kelime sözlüğü boyutu:", len(v.vocabulary_))
c = char_tfidf().fit(X_tr)
print("Karakter parçası sözlüğü boyutu:", len(c.vocabulary_))

# Örnek: bir belgenin en yüksek ağırlıklı kelimeleri
ex = train.sample(1, random_state=SEED).iloc[0]
vec = v.transform([ex.text_norm]).toarray().ravel()
top = np.argsort(vec)[::-1][:10]
names = v.get_feature_names_out()
print(f"\nÖrnek belge ({ex.label}) – en yüksek TF-IDF ağırlıkları:")
for i in top:
    if vec[i] > 0:
        print(f"  {names[i]:25s} {vec[i]:.3f}")

## 4. Modeller ve hiperparametre seçimi (validation ile)

| Model | Fikir | Ayarlanan parametre |
|---|---|---|
| Multinomial Naive Bayes | Kelime olasılıklarından Bayes kuralı; çok hızlı | `alpha` (düzgünleştirme) |
| Logistic Regression | Doğrusal model, sınıf olasılığı verir | `C` (düzenlileştirme) |
| Linear SVM | Sınıfları en geniş aralıkla ayıran doğrusal sınır | `C` |

Her model × özellik çifti için en iyi parametre **validation macro-F1**'e göre seçilir.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

MODELS_GRID = {
    "Naive Bayes":         (lambda p: MultinomialNB(alpha=p), [0.01, 0.05, 0.1, 0.5, 1.0]),
    "Logistic Regression": (lambda p: LogisticRegression(C=p, max_iter=3000, random_state=SEED), [0.5, 1, 5, 10, 50]),
    "Linear SVM":          (lambda p: LinearSVC(C=p, random_state=SEED), [0.05, 0.1, 0.5, 1, 5]),
}
FEATURES = {"kelime": word_tfidf, "karakter": char_tfidf}

rows, best = [], {}
for fname, fmake in FEATURES.items():
    for mname, (mmake, grid) in MODELS_GRID.items():
        for p in grid:
            pipe = Pipeline([("tfidf", fmake()), ("clf", mmake(p))])
            t0 = time.time(); pipe.fit(X_tr, y_tr); dt = time.time() - t0
            pred = pipe.predict(X_va)
            r = {"özellik": fname, "model": mname, "parametre": p,
                 "val_acc": accuracy_score(y_va, pred), "val_macro_f1": f1_score(y_va, pred, average="macro"),
                 "eğitim_sn": dt}
            rows.append(r)
            key = (fname, mname)
            if key not in best or r["val_macro_f1"] > best[key]["val_macro_f1"]:
                best[key] = {**r, "pipe": pipe}

grid_df = pd.DataFrame(rows)
grid_df.to_csv(ROOT / "hiperparametre_arama.csv", index=False)
best_val = pd.DataFrame([{k: v for k, v in b.items() if k != "pipe"} for b in best.values()])
best_val.sort_values("val_macro_f1", ascending=False).round(4)

## 5. Test sonuçları (görülmemiş belgeler)

En iyi parametreleriyle 6 model (3 algoritma × 2 özellik) **test** setinde değerlendirilir.

| Metrik | Anlamı |
|---|---|
| Accuracy | Doğru tahmin oranı |
| Precision | "X dediklerinin" kaçı gerçekten X |
| Recall | Gerçek X'lerin kaçını bulabildi |
| F1 | Precision ve recall'un harmonik ortalaması |
| Macro ortalama | 4 sınıfın eşit ağırlıklı ortalaması |

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def evaluate(pipe, X, y):
    pred = pipe.predict(X)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, pred), "precision": p, "recall": r, "f1": f}, pred

readable = ~test.is_empty.values
results, preds = [], {}
for (fname, mname), b in best.items():
    m_all, pred = evaluate(b["pipe"], X_te, y_te)
    m_read, _ = evaluate(b["pipe"], X_te[readable], y_te[readable])
    preds[(fname, mname)] = pred
    results.append({"model": mname, "özellik": fname, "parametre": b["parametre"],
                    **{f"test_{k}": v for k, v in m_all.items()},
                    "okunabilir_test_f1": m_read["f1"]})

res_df = pd.DataFrame(results).sort_values("test_f1", ascending=False).reset_index(drop=True)
res_df.to_csv(ROOT / "test_sonuclari.csv", index=False)
res_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
lab = res_df["model"] + "\n(" + res_df["özellik"] + ")"
x = np.arange(len(res_df)); w = 0.2
for i, m in enumerate(["test_accuracy", "test_precision", "test_recall", "test_f1"]):
    ax.bar(x + (i - 1.5) * w, res_df[m], w, label=m.replace("test_", ""))
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=9)
ax.set_ylim(max(0, res_df[["test_accuracy", "test_f1"]].min().min() - 0.15), 1.0)
ax.set_ylabel("Skor (test)"); ax.legend(ncol=4, loc="lower right"); ax.grid(axis="y", alpha=0.3)
ax.set_title("Modellerin test performansı")
plt.tight_layout(); plt.savefig(FIG / "model_karsilastirma.png", dpi=130); plt.show()

## 6. Confusion matrix (karmaşıklık matrisleri)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

order = [(r["özellik"], r["model"]) for _, r in res_df.iterrows()]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, key in zip(axes.ravel(), order):
    cm = confusion_matrix(y_te, preds[key], labels=CLASSES)
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
    f1 = res_df[(res_df.özellik == key[0]) & (res_df.model == key[1])].test_f1.iloc[0]
    ax.set_title(f"{key[1]} · {key[0]}\nmacro-F1 = {f1:.3f}", fontsize=11)
    ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")
plt.tight_layout(); plt.savefig(FIG / "confusion_matrices.png", dpi=120); plt.show()

## 7. Hangi algoritma hangi belge türünde başarılı / başarısız?

Sınıf bazında F1 skorları. Satırlar belge türü, sütunlar model.

In [ ]:
per_class = {}
for key in order:
    _, _, f, _ = precision_recall_fscore_support(y_te, preds[key], labels=CLASSES, zero_division=0)
    per_class[f"{key[1]} ({key[0]})"] = f
pc_df = pd.DataFrame(per_class, index=CLASSES).round(3)
pc_df.to_csv(ROOT / "sinif_bazli_f1.csv")
display(pc_df.style.background_gradient(cmap="RdYlGn", axis=None, vmin=pc_df.values.min(), vmax=1))

print("\nEn kolay sınıf (ortalama F1):", pc_df.mean(axis=1).idxmax())
print("En zor sınıf (ortalama F1):", pc_df.mean(axis=1).idxmin())

In [ ]:
# OCR kalitesine göre başarı: iyi okunan belgelerde model daha mı başarılı?
best_key = order[0]
test_eval = test.assign(pred=preds[best_key], dogru=lambda d: d.pred == d.label)
print(f"En iyi model: {best_key[1]} ({best_key[0]})")
display(test_eval.groupby("ocr_quality")["dogru"].agg(belge="count", dogruluk="mean").reindex(["iyi", "orta", "zayıf"]).round(3))

**Yorum için ipuçları** (raporun "Bulgular" bölümü):
- Hangi iki sınıf en çok karışıyor? (confusion matrix'te köşegen dışı en büyük hücre)
- Karakter TF-IDF, kelime TF-IDF'ten iyi mi? OCR hataları ne kadar etkiliyor?
- Naive Bayes hangi sınıfta diğerlerinden geri kalıyor?
- OCR kalitesi "zayıf" olan belgelerde doğruluk ne kadar düşüyor?

## 8. Modeller neye bakıyor? Sınıf başına en etkili kelimeler

In [ ]:
lr_key = ("kelime", "Logistic Regression")
pipe = best[lr_key]["pipe"]
names = pipe.named_steps["tfidf"].get_feature_names_out()
coefs = pipe.named_steps["clf"].coef_
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, cls, w in zip(axes, pipe.named_steps["clf"].classes_, coefs):
    top = np.argsort(w)[::-1][:12]
    ax.barh(names[top][::-1], w[top][::-1], color="#3b6fb6")
    ax.set_title(f"{cls}")
plt.suptitle("Logistic Regression (kelime TF-IDF): sınıfı en çok destekleyen kelimeler")
plt.tight_layout(); plt.savefig(FIG / "etkili_kelimeler.png", dpi=120); plt.show()

## 9. Yanlış sınıflandırılan belgeler

In [ ]:
from PIL import Image
wrong = test_eval[~test_eval.dogru]
print(f"Yanlış sınıflandırılan: {len(wrong)} / {len(test_eval)}")
display(pd.crosstab(wrong.label, wrong.pred).rename_axis(index="gerçek", columns="tahmin"))

sample = wrong.sample(min(6, len(wrong)), random_state=SEED)
if len(sample):
    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 6))
    for ax, (_, r) in zip(np.atleast_1d(axes), sample.iterrows()):
        ax.imshow(Image.open(DATA / r["path"]), cmap="gray"); ax.axis("off")
        ax.set_title(f"gerçek: {r.label}\ntahmin: {r.pred}\nOCR: {r.ocr_quality}", fontsize=10)
    plt.tight_layout(); plt.savefig(FIG / "yanlis_siniflandirilanlar.png", dpi=110); plt.show()

## 10. Şüpheli etiketler (confident learning)

RVL-CDIP'te belgelerin yaklaşık %8'inin yanlış etiketli olduğu bilinir (Larson ve ark., EACL 2023).
Train belgeleri 5 katlı çapraz doğrulama ile tahmin edilir; modelin **çok emin olup etiketle çeliştiği** belgeler listelenir.
Bu listeyi görsellere bakarak elle kontrol edin ve gerçekten yanlış etiketli olanları `yanlis_etiket.txt` dosyasına (her satıra bir `path`) yazın.

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold

lr_pipe = Pipeline([("tfidf", word_tfidf()),
                    ("clf", LogisticRegression(C=best[lr_key]["parametre"], max_iter=3000, random_state=SEED))])
proba = cross_val_predict(lr_pipe, X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")
cls_names = sorted(CLASSES)
train["cv_pred"] = [cls_names[i] for i in proba.argmax(1)]
train["cv_conf"] = proba.max(1)
train["etiket_olasılığı"] = proba[np.arange(len(train)), [cls_names.index(l) for l in train.label]]

suspect = train[(train.cv_pred != train.label) & (train.cv_conf >= 0.8)].sort_values("cv_conf", ascending=False)
suspect[["path", "label", "cv_pred", "cv_conf", "etiket_olasılığı"]].to_csv(ROOT / "supheli_etiketler.csv", index=False)
print(f"Şüpheli belge: {len(suspect)} ({100 * len(suspect) / len(train):.1f}% of train)")
display(pd.crosstab(suspect.label, suspect.cv_pred).rename_axis(index="etiket", columns="model diyor"))

In [ ]:
# Şüphelileri görsel olarak incele (sayfa sayfa: page = 0, 1, 2 ...)
page = 0
chunk = suspect.iloc[page * 8:(page + 1) * 8]
if len(chunk):
    fig, axes = plt.subplots(2, 4, figsize=(18, 12))
    for ax in axes.ravel(): ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), chunk.iterrows()):
        ax.imshow(Image.open(DATA / r["path"]), cmap="gray")
        ax.set_title(f"{Path(r.path).name}\netiket: {r.label} · model: {r.cv_pred} ({r.cv_conf:.0%})", fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print("Bu sayfada belge yok.")

### 10b. Yanlış etiketlileri çıkarıp yeniden eğitme (isteğe bağlı)

`yanlis_etiket.txt` dosyasını oluşturduysanız aşağıdaki hücre bu belgeleri **sadece train'den** çıkarıp en iyi modeli yeniden eğitir ve test sonucunu karşılaştırır. Test seti değişmez.

In [ ]:
bad_file = ROOT / "yanlis_etiket.txt"
if bad_file.exists():
    bad = {l.strip() for l in bad_file.read_text().splitlines() if l.strip()}
    clean_train = train[~train.path.isin(bad)]
    fname, mname = best_key
    p = best[best_key]["parametre"]
    pipe2 = Pipeline([("tfidf", FEATURES[fname]()), ("clf", MODELS_GRID[mname][0](p))]).fit(clean_train.text_norm, clean_train.label)
    before, _ = evaluate(best[best_key]["pipe"], X_te, y_te)
    after, _ = evaluate(pipe2, X_te, y_te)
    print(f"Çıkarılan train belgesi: {len(train) - len(clean_train)}")
    display(pd.DataFrame({"önce": before, "sonra": after}).round(4))
else:
    print("yanlis_etiket.txt yok; bu adım atlandı.")

## 11. En iyi modeli kaydet (05 demo için)

In [ ]:
import joblib
final = best[best_key]["pipe"]
joblib.dump(final, MODELS / "best_model.joblib")
info = {"model": best_key[1], "özellik": best_key[0], "parametre": best[best_key]["parametre"],
        "test_macro_f1": float(res_df.test_f1.iloc[0]), "test_accuracy": float(res_df.test_accuracy.iloc[0])}
(MODELS / "best_model.json").write_text(json.dumps(info, ensure_ascii=False, indent=2))
print(json.dumps(info, ensure_ascii=False, indent=2))

## Çıktılar

| Dosya | İçerik |
|---|---|
| `test_sonuclari.csv` | 6 modelin accuracy / precision / recall / F1 tablosu |
| `sinif_bazli_f1.csv` | Sınıf × model F1 tablosu |
| `hiperparametre_arama.csv` | Tüm denemeler (validation) |
| `supheli_etiketler.csv` | Elle kontrol edilecek şüpheli etiketler |
| `figures/model_karsilastirma.png`, `confusion_matrices.png`, `etkili_kelimeler.png`, `yanlis_siniflandirilanlar.png` | Rapor görselleri |
| `models/best_model.joblib` | 05 demo |

Sonraki adım → **05_end_to_end_demo.ipynb**